# 03 Covariance construction from a reflectivity profile

This notebook confirms how a height reflectivity profile maps to an interferometric covariance matrix, the central object behind the COMET-style covariance-matching term. The covariance is `C = integral over x of S(x) a(x) a(x)^H dx`, which the check forms as an einsum of `geom.outer` with the profile.

Modules exercised: `tools.tomo_geometry.TomoGeometry`, `tools.gaussians.GaussianMixture`, and the covariance einsum used inside `PhysicsComponents.covariance_matching` and `capon_cycle`.

We show that a narrow scatterer gives a near-rank-one, highly coherent covariance, while a broad scatterer decorrelates the off-diagonal entries.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from configuration.training_config import GeometryConfig
from tools.tomo_geometry            import TomoGeometry
from tools.gaussians                import GaussianMixture

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
xt     = torch.tensor(x_axis, device=DEVICE)
geom   = TomoGeometry(GeometryConfig(), xt)

def covariance(profile):
    prof_t = to_check_tensor(profile[None, :])
    cov    = torch.einsum("ijk,bkhw->bijhw", geom.outer, prof_t.to(geom.outer.dtype)) * dx
    return cov[0, :, :, 0, 0]


## Narrow versus broad scatterer

A point-like scatterer at a single height produces a rank-one covariance: the off-diagonal magnitudes equal the diagonal (full coherence). A vertically extended scatterer averages many phase ramps, so the off-diagonals collapse toward zero for widely separated tracks.

In [ ]:
narrow = gaussian_profile(x_axis, amp=1.0, mu=10.0, sigma=0.8)
broad  = gaussian_profile(x_axis, amp=1.0, mu=10.0, sigma=8.0)

cov_narrow = covariance(narrow)
cov_broad  = covariance(broad)

fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.3))
for ax, cov, title in zip(axes, [cov_narrow, cov_broad], ["narrow (sigma=0.8)", "broad (sigma=8)"]):
    im = ax.imshow(cov.abs().cpu().numpy(), cmap="viridis")
    ax.set_title(f"|C| {title}")
    ax.set_xlabel("track")
    ax.set_ylabel("track")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## Coherence magnitude versus track separation

Normalising each off-diagonal by the diagonal gives the interferometric coherence. We plot its magnitude against track index relative to the reference track to expose the decorrelation.

In [ ]:
def coherence_row(cov):
    d   = torch.sqrt(torch.diagonal(cov).real.clamp(min=1e-12))
    gam = cov[0, :] / (d[0] * d)
    return gam.abs().cpu().numpy()

fig, ax = plt.subplots(figsize=(6.5, 3.3))
ax.plot(coherence_row(cov_narrow), "o-", color="C0", label="narrow")
ax.plot(coherence_row(cov_broad),  "s-", color="C1", label="broad")
ax.set_xlabel("track index")
ax.set_ylabel("|coherence| vs track 0")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Coherence decay with vertical extent")
ax.legend()
plt.show()

## Expected outcome

The narrow scatterer yields a covariance whose magnitude is uniform across all entries (coherence near one for every track pair), while the broad scatterer shows a strong diagonal with rapidly decaying off-diagonals. This is the physical signal the covariance- matching term must preserve when a Gaussian fit replaces the Capon profile.